# Simulating Discrete Traits (Markov Model)

This notebook demonstrates `toytree.pcm.simulate_discrete_trait()` to simulate a multi-state trait on a tree under a continuous time Markov Chain (CTMC) model. 


In [2]:
import toytree
import numpy as np

tree = toytree.rtree.bdtree(12, seed=123)

## Quick Example

In [3]:
# simulate a discrete trait and store to the tree as feature "X"
trait = tree.pcm.simulate_discrete_trait(nstates=3, name="X", seed=123, inplace=True)

# draw the trait on the tree's nodes
tree.draw(node_sizes=10, node_mask=False, node_colors="X");

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="298.71999999999997px" viewBox="0 0 300.0 298.71999999999997" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="ta7f103c1425e4fec9a6dedef10ec0a7e"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11

## Continuous-time Markov Chain

`simulate_discrete_trait()` uses a continuous-time Markov chain (CTMC)
to model transitions among a fixed number of discrete states along
each branch. The model is defined by an instantaneous rate matrix
**Q** where each off-diagonal entry `q_ij` is the rate of change from
state *i* to state *j*. Diagonal entries are set so each row sums
to zero, and transition probabilities over time *t* are computed as
`P(t) = expm(Q * t)`.

To use this method you must specify the number of states (``states``), and you can additionally
specify a ``model`` that constrains how the rate matrix **Q** can be parameterized. The 
supported model types are:

- **ER (equal rates)**: all off-diagonal rates are the same value.
  This is the simplest model and assumes every state changes to every
  other state at the same rate.
- **SYM (symmetric rates)**: rates are symmetric (`q_ij = q_ji`),
  but different state pairs can have different rates. This allows
  variation among pairs while keeping reversibility.
- **ARD (all rates different)**: every off-diagonal rate can be
  different (`q_ij` and `q_ji` may differ). This is the most flexible
  model, allowing directional biases between states.

In all cases, you can provide `relative_rates` and `state_frequencies`
explicitly, or let `toytree` sample valid parameters for the chosen
model. The `rate_scalar` multiplies the relative rates to set the
overall tempo of change.

## Simulating a trait

A simulated trait is returned as a ``pandas.Series`` mapping of node idx label to a discrete trait.

In [8]:
# simulate a discrete trait on a tree
trait = tree.pcm.simulate_discrete_trait(nstates=3, name="X", seed=123)

# return is a pandas.Series object
trait

0     A
1     A
2     B
3     C
4     C
5     C
6     C
7     C
8     B
9     A
10    A
11    A
12    A
13    A
14    C
15    C
16    C
17    C
18    A
19    A
20    A
21    A
22    A
Name: X, dtype: object

### Inplace

Use ``inplace`` to store the trait directly to the tree. This is the recommended workflow for using a simulated feature in downstream plotting or analysis tools. 

Note that even when ``inplace=True`` the Series is still returned by the function call in addition to storing it to the tree.

In [9]:
tree.pcm.simulate_discrete_trait(nstates=3, name="X", seed=123, inplace=True)
tree.features

('idx', 'name', 'height', 'dist', 'support', 'X')

### tips_only

Use ``tips_only`` to simulate traits only for tip nodes in the tree. Internal nodes will not be assigned a value, and if a value already exists for the simulated feature name when ``inplace=True`` it will be overwritten as NaN. 

In [10]:
# simulate only tip data
trait = tree.pcm.simulate_discrete_trait(nstates=3, name="X", seed=123, inplace=True, tips_only=True)

# plot data 
tree.draw(node_sizes=10, node_mask=False, node_colors="X");

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="298.71999999999997px" viewBox="0 0 300.0 298.71999999999997" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t387a4d1b5e344952b1d9a94b9ae8ebf1"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11

### relative_rates
This parameter is used to enter off-diagonal entries for the instantaneous rate matrix **Q**. Larger
values increase transition frequency between specific state pairs. 

If you enter an invalid relative rates matrix given the designated model a ``ToyTreeError`` will be raised.

In [12]:
# transitions 0->1 are 6X more likely than 1->0
rates = [[0, 3.0], [0.5, 0]]

# simulate traits
tree.pcm.simulate_discrete_trait(nstates=2, model="ARD", relative_rates=rates, name="rates_bias", seed=321, inplace=True)
tree.draw(node_sizes=10, node_mask=False, node_colors="rates_bias");

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="298.71999999999997px" viewBox="0 0 300.0 298.71999999999997" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t70836302ea1443778df346ffea7a9b13"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11

### state_frequencies

Stationary frequencies (equilibrium proportions) that influence long-run state occupancy and, for reversible models, how rates are scaled among states.

In [13]:
# state 0 is more common than state 1 at equilibrium
freqs = [0.9, 0.1]

# simulate traits
tree.pcm.simulate_discrete_trait(nstates=2, model="SYM", state_frequencies=freqs, name="freqs_bias", seed=321, inplace=True)
tree.draw(node_sizes=10, node_mask=False, node_colors="freqs_bias");

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="298.71999999999997px" viewBox="0 0 300.0 298.71999999999997" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t2dbf43a30d3143e48ac7e881c3412dea"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11

### root_state

Fixes the starting state at the root. This biases the entire simulation toward descendants of that root state.

In [15]:
# simulate traits
tree.pcm.simulate_discrete_trait(nstates=2, model="ER", root_state=0, name="root_0", seed=321, inplace=True)
tree.draw(node_sizes=10, node_mask=False, node_colors="root_0");

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="298.71999999999997px" viewBox="0 0 300.0 298.71999999999997" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t4fcd2fb3bacc4c06a0da3c6bcdabdcb6"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11

### rate_scalar

Multiplies the relative rates, controlling the overall tempo of evolution (higher values -> more changes).

In [16]:
# simulate traits
tree.pcm.simulate_discrete_trait(nstates=2, model="ER", rate_scalar=0.2, name="slow", seed=321, inplace=True)
tree.draw(node_sizes=10, node_mask=False, node_colors="slow");

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="298.71999999999997px" viewBox="0 0 300.0 298.71999999999997" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="td89de45e4e584ff582a72c9f0cf4013c"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11

## Other options

### seed
Set the random generator seed.

### name
You can set the name of the Series, which is also the feature name stored when ``inplace=True``.

In [17]:
# simulate traits
trait = tree.pcm.simulate_discrete_trait(nstates=2, name="trait")
trait.name

'trait'

### state_names
Set the names of character states. This must match the ``nstates`` arg. The default is A-Z if there are <26 states, else numeric string states. You can set states manually.

In [18]:
# simulate traits
trait = tree.pcm.simulate_discrete_trait(nstates=2, state_names=["forest", "desert"], name="ecotype")
trait.head(10)

0    forest
1    desert
2    desert
3    desert
4    desert
5    forest
6    forest
7    forest
8    desert
9    forest
Name: ecotype, dtype: object

## See Also

- ancestral state reconstruction
- pglm
- color-mapping
- stochastic mapping